# DataCite citations

Run this either everytime new DataCite datasets are pulled or to update citations of older datasets.

The DataCite metadata comes with citations to datasets, if any (DOI only).
Here we process them to get their publication year and generate the citation file with weights

In [21]:
%load_ext autoreload
%autoreload 2
from sindex.sources.datacite.jobs import (
    batch_find_citations_from_dc_serial,
    extract_unique_dois_from_citation_blocks,
    lookup_dates_in_oa_snapshot,
)
from sindex.sources.datacite.utils import get_citation_blocks_from_ndjson

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Processing citations of newly pulled DataCite datasets 

### Extract citation blocks from slim metadata files for easier processing

In [2]:
datacite_slim_dir = r"D:\may-2026-data\records\slim-records\datacite-slim-records"
citation_blocks_file = r"D:\may-2026-data\citations\datacite\citation_blocks.ndjson"

In [4]:
get_citation_blocks_from_ndjson(datacite_slim_dir, citation_blocks_file)

[2026-06-01 18:01:29.091483] Processing 213 files...
[213/213] slim-99.ndjson â€” 96,968 saved so farr
[2026-06-01 18:06:43.018949] Complete! Total Records Saved: 96,968


### Get a list of unique citing DOIs for efficiently querying their publication year

In [8]:
citations_parquet_file = (
    r"D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet"
)

In [10]:
extract_unique_dois_from_citation_blocks(citation_blocks_file, citations_parquet_file)

Scanning citation_blocks.ndjson...
Scanned 96,968 records. Unique DOIs: 50,876
Exporting 50,876 unique DOIs to D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet...
[SUCCESS] Saved to D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet


### Get publication dates from OpenAlex snapshot if available

In [15]:
# Paths
citations_parquet_file_with_pubdates = (
    r"D:\may-2026-data\citations\datacite\datacite_citation_dois_with_pubdates.parquet"
)
oa_db_path = r"C:\Users\BPatel\Documents\oa_duckdb_fast\oa_snapshot.duckdb"

In [14]:
lookup_dates_in_oa_snapshot(
    oa_db_path, citations_parquet_file, citations_parquet_file_with_pubdates
)

Joining D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet with OpenAlex database...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    [SUCCESS] Found 19,376 matches.
    [INFO] Results saved to: D:\may-2026-data\citations\datacite\datacite_citation_dois_with_pubyears.parquet
    [INFO] Time taken: 30.27s


### Create citations file

In [17]:
out_ndjson = r"D:\may-2026-data\citations\datacite\dc_citations.ndjson"

In [20]:
batch_find_citations_from_dc_serial(
    citation_blocks_file, out_ndjson, citations_parquet_file_with_pubdates
)

[2026-06-01 18:55:16.786371] Starting SERIAL processing...
[*] Loading cache from datacite_citation_dois_with_pubdates.parquet...
    -> Loaded 19,376 dates into memory.
[*] Counting exact lines in input file...
    -> Total workload: 96,968 lines.
[*] Processing lines...
Progress: 99.00% | Found: 112,722

[DONE] Finished in 68190.78s.
       Total Lines Processed: 96,968
       Total Citations Found: 113,883


In [18]:
batch_find_citations_from_dc_parallel(
    citation_blocks_file, out_ndjson, citations_parquet_file_with_pubdates
)

[*] Loading cache from datacite_citation_dois_with_pubdates.parquet...
    -> Loaded 19,376 dates into memory.
[*] Counting exact lines in input file...
    -> Total workload: 96,968 lines.
[*] Using 32 workers
[*] Launching workers...
Progress: 3.30% | Processed: 3,200/96,968

HTTPError: 429 Client Error: Too Many Requests for url: https://api.datacite.org/dois/10.1007/jhep11(2025)076

## Process citations of previously saved DataCite datasets